# 00 — Setup & Smoke Test

Quickstart for the Graph XAI pipeline. Confirms that `src/xai/` is wired up correctly against the trained checkpoints and that **both architectures** produce sensible output before running the heavyweight population notebooks (`01_sg_population.ipynb`, `02_st_population.ipynb`).

What this notebook does:
1. Environment + version check
2. Channel-layout sanity
3. **SG smoke** — fold 1 of `kfold-5 mt2 hbo` with GNNExplainer at `epochs=20` (smoke speed; SPEC default is 200)
4. **ST smoke** — fold 1 of the same cell with native attention (essentially free)
5. Cross-architecture sanity (Spearman ρ on a single fold — diagnostic only; population-level ρ is computed in `03_cross_arch_comparison.ipynb`)

Reference: [`docs/SPEC_xai_graph.md`](../../../docs/SPEC_xai_graph.md) (rev. 4).

In [1]:
%matplotlib inline
import os, sys
from pathlib import Path

import numpy as np
import torch

# Notebook lives at src/notebook/xai/<this>.ipynb → project root is 3 levels up.
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xai import (
    XAIRunConfig,
    discover_checkpoints, load_checkpoint,
    explain_sg_checkpoint, explain_st_checkpoint,
    aggregate_population,
    plot_montage_channel_importance, plot_pair_matrix,
    plot_temporal_attention,
    plot_sg_vs_st_scatter, plot_pair_matrix_diff,
    write_run_json,
    CHANNEL_NAMES, GRID_POS, N_CH,
)
import torch_geometric, captum

print(f"PROJECT_ROOT     = {PROJECT_ROOT}")
print(f"torch:             {torch.__version__}")
print(f"torch_geometric:   {torch_geometric.__version__}")
print(f"captum:            {captum.__version__}")
print(f"CUDA available:    {torch.cuda.is_available()}")
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"DEVICE             = {DEVICE}")

PROJECT_ROOT     = /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method
torch:             2.6.0+cu124
torch_geometric:   2.7.0
captum:            0.9.0
CUDA available:    True
DEVICE             = cuda:0


## 1. Channel layout sanity

`CHANNEL_NAMES` and `GRID_POS` are copied verbatim from `src/notebook/statistical-analysis/04_severity_correlation/04_severity_correlation.ipynb` so XAI heatmaps line up channel-for-channel with the statistical-analysis figures.

In [2]:
print(f"N_CH = {N_CH}  (expected 23)")
print()
print("CHANNEL_NAMES:")
for i in range(0, N_CH, 6):
    print("  " + ", ".join(CHANNEL_NAMES[i:i+6]))
print()
print(f"GRID_POS first 3: {GRID_POS[:3]}  ...  GRID_POS last: {GRID_POS[-1]}")
print(f"All grid positions in 5×7 bounds: "
      f"{all(0 <= r < 5 and 0 <= c < 7 for r, c in GRID_POS)}")

N_CH = 23  (expected 23)

CHANNEL_NAMES:
  S1_D1, S1_D3, S2_D2, S2_D1, S2_D5, S3_D1
  S3_D3, S3_D4, S3_D6, S4_D4, S4_D5, S4_D7
  S5_D2, S5_D5, S5_D8, S6_D3, S6_D6, S7_D4
  S7_D6, S7_D7, S8_D5, S8_D7, S8_D8

GRID_POS first 3: [(0, 2), (1, 1), (0, 4)]  ...  GRID_POS last: (4, 5)
All grid positions in 5×7 bounds: True


## 2. SG smoke run — `FlexibleGATNet`, GNNExplainer

One fold of SG `kfold-5 mt2 hbo` with `gnn_explainer_epochs=20` for speed. Writes the §7.3 deliverables (`node_importance.csv`, `edge_importance.csv`, `channel_pair_matrix.npy`, `feature_importance.csv`, `result_meta.json`) plus `run.json` to `research/xai/_smoke/sg_fold1/`.

**Expected runtime:** ~30s on GPU, ~2–3 min on CPU.

> **Note:** `cfg.estimator='gnn'` is the primary path. `01_sg_population.ipynb` runs the full 6-cell sweep with all three estimators (GNNExplainer + CaptumExplainer-IG + AttentionExplainer) for the C4 acceptance check.

In [ ]:
SG_EXPERIMENT_ROOT = PROJECT_ROOT / "research/experiments/20260506/leak-free-patience-9999/spatial-graph"
SG_SMOKE_OUT = PROJECT_ROOT / "research/xai/_smoke/sg_fold1"
SG_SMOKE_OUT.mkdir(parents=True, exist_ok=True)

sg_cfg = XAIRunConfig(
    arch="sg", hb="hbo", regime="kfold-5", mt=2,
    experiment_root=str(SG_EXPERIMENT_ROOT),
    output_dir=str(SG_SMOKE_OUT),
    device=DEVICE,
    gnn_explainer_epochs=20,
    gnn_explainer_lr=0.01,
    estimator="gnn",
    seed=42,
)
sg_infos = discover_checkpoints(SG_EXPERIMENT_ROOT, arch="sg", hb="hbo", regime="kfold-5", mt=2)
sg_fold1 = next(i for i in sg_infos if i.fold == 1)
print(f"SG checkpoints discovered: {len(sg_infos)} (using {sg_fold1.label})")

sg_loaded = load_checkpoint(sg_fold1, sg_cfg)
sg_trials = explain_sg_checkpoint(sg_loaded, sg_cfg)
sg_result = aggregate_population(sg_trials, arch="sg", hb="hbo", regime="kfold-5", mt=2)
sg_result.to_csv(SG_SMOKE_OUT)
write_run_json(sg_cfg, sg_result, SG_SMOKE_OUT)

top5_idx = (-sg_result.channel_importance_mean).argsort()[:5]
print(f"\nSG fold 1 — n_trials={sg_result.n_trials} included / {sg_result.n_trials_total} total "
      f"({sg_result.included_pct:.1f}%)")
print(f"            n_subjects={sg_result.n_subjects}")
print(f"            top-5 channels: {[CHANNEL_NAMES[i] for i in top5_idx]}")

In [ ]:
# Render figures + display inline.
plot_montage_channel_importance(sg_result, SG_SMOKE_OUT)
plot_pair_matrix(sg_result, SG_SMOKE_OUT)

from IPython.display import Image, display
display(Image(str(SG_SMOKE_OUT / "fig_montage_channel_importance.png")))
display(Image(str(SG_SMOKE_OUT / "fig_pair_matrix.png")))

## 3. ST smoke run — `WindowedSpatioTemporalGATNet`, native attention

ST native attention is essentially free at inference: no per-trial optimisation, just `model.explain()` extracting GATv2 attention + the additive temporal `α_k`. Reductions follow SPEC §6.1 (heads → layers → α_k weighting → symmetric pair matrix).

**Expected runtime:** <1 min.

In [ ]:
ST_EXPERIMENT_ROOT = PROJECT_ROOT / "research/experiments/20260501/spatial_temporal_graph"
ST_SMOKE_OUT = PROJECT_ROOT / "research/xai/_smoke/st_fold1"
ST_SMOKE_OUT.mkdir(parents=True, exist_ok=True)

st_cfg = XAIRunConfig(
    arch="st", hb="hbo", regime="kfold-5", mt=2,
    experiment_root=str(ST_EXPERIMENT_ROOT),
    output_dir=str(ST_SMOKE_OUT),
    device=DEVICE,
    head_reduce="mean",
    layer_reduce="mean",
    seed=42,
)
st_infos = discover_checkpoints(ST_EXPERIMENT_ROOT, arch="st", hb="hbo", regime="kfold-5", mt=2)
st_fold1 = next(i for i in st_infos if i.fold == 1)
print(f"ST checkpoints discovered: {len(st_infos)} (using {st_fold1.label})")

st_loaded = load_checkpoint(st_fold1, st_cfg)
st_trials = explain_st_checkpoint(st_loaded, st_cfg)
st_result = aggregate_population(st_trials, arch="st", hb="hbo", regime="kfold-5", mt=2)
st_result.to_csv(ST_SMOKE_OUT)
write_run_json(st_cfg, st_result, ST_SMOKE_OUT)

st_top5_idx = (-st_result.channel_importance_mean).argsort()[:5]
print(f"\nST fold 1 — n_trials={st_result.n_trials} included / {st_result.n_trials_total} total "
      f"({st_result.included_pct:.1f}%)")
print(f"            n_subjects={st_result.n_subjects}")
print(f"            n_windows={st_result.temporal_attention_mean.shape[0]}")
print(f"            top-5 channels: {[CHANNEL_NAMES[i] for i in st_top5_idx]}")

In [ ]:
# Render figures + display inline.
plot_montage_channel_importance(st_result, ST_SMOKE_OUT)
plot_pair_matrix(st_result, ST_SMOKE_OUT)
plot_temporal_attention(st_result, ST_SMOKE_OUT)

from IPython.display import Image, display
display(Image(str(ST_SMOKE_OUT / "fig_montage_channel_importance.png")))
display(Image(str(ST_SMOKE_OUT / "fig_pair_matrix.png")))
display(Image(str(ST_SMOKE_OUT / "fig_temporal_attention.png")))

## 4. Cross-architecture sanity

Comparing one SG fold to one ST fold is **not** the SPEC §11 C4/C6 acceptance check — those are population-level. This cell is a fast diagnostic: if Spearman ρ between SG and ST channel-importance rankings on the same fold is wildly negative, something is broken upstream and the population notebooks aren't worth running yet.

In [ ]:
from scipy import stats

CROSS_OUT = PROJECT_ROOT / "research/xai/_smoke/cross_arch"
CROSS_OUT.mkdir(parents=True, exist_ok=True)

plot_sg_vs_st_scatter(sg_result, st_result, CROSS_OUT)
plot_pair_matrix_diff(sg_result, st_result, CROSS_OUT)

rho, _ = stats.spearmanr(sg_result.channel_importance_mean, st_result.channel_importance_mean)
print(f"SG-vs-ST channel-importance Spearman ρ on fold 1 only: ρ = {rho:+.3f}")
print()
print("Population-level acceptance criteria (SPEC §11) are evaluated in 03_cross_arch_comparison.ipynb")
print("  C2: ρ between any two folds in same regime ≥ 0.4")
print("  C3: ρ(mt=2 ranking, mt=4 ranking) ≥ 0.5")
print("  C4: ρ between any two of {GNNExplainer, Captum-IG, AttentionExplainer} top-10 channels ≥ 0.4 (SG)")
print("  C5: ρ(native attention top-10, GNN object-mask top-10) ≥ 0.4 (ST)")
print("  C6: ≥ 2 of {S1_D1, S5_D5, S3_D3, S2_D1, S4_D5, S4_D7} in top-10 (advisory)")

from IPython.display import Image, display
display(Image(str(CROSS_OUT / "fig_sg_vs_st_scatter.png")))
display(Image(str(CROSS_OUT / "fig_pair_matrix_diff.png")))

## 5. Done

If every cell above ran cleanly, the XAI infrastructure is wired up correctly. Next steps:

- **Population sweeps:** run `01_sg_population.ipynb` (3 regimes × 2 mt × 3 estimators = 18 SG runs) and `02_st_population.ipynb` (3 regimes × 2 mt × {primary, supplementary} = 12 ST runs).
- **Cross-arch comparison:** `03_cross_arch_comparison.ipynb` consumes the saved CSVs.
- **Outputs:** all CSVs and figures land under `research/xai/`.

> Smoke-run outputs above sit at `research/xai/_smoke/` and can be deleted once the population sweeps complete.